# Agent的高级用法

## 1、设置agent的名称

使用字段name来进行设置

举例：

In [5]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from dotenv import load_dotenv
from rich import print as rprint
import os

MODEL_NAME = "deepseek-flash"

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2.模型的初始化
model = init_chat_model(
    model="deepseek:deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,

)

agent = create_agent(
    model=model,
    name="agent01"
)


# 调用行为
response = agent.invoke({ # 调用agent的时候invoke里面是以字典的方式传进来
    "messages":[
        {"role":"system","content":"你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题"},
        {"role":"user","content":"100 + 20 * 3 =?"}
    ]
})


# rprint(response)

for message in response["messages"]:
    message.pretty_print()

================================ System Message ================================

你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题
================================ Human Message =================================

100 + 20 * 3 =?
================================== Ai Message ==================================
Name: agent01

这个算式是：

100 + 20 × 3

按照运算顺序，先算乘法：

20 × 3 = 60

再算加法：

100 + 60 = 160

所以：

**100 + 20 × 3 = 160**


## 2、系统提示词的设置

使用 system_prompt 设置，定义 Agent 行为。这个参数可以是 str或者 SystemMessage类型

In [4]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2.模型的初始化
model = init_chat_model(
    model="deepseek:deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,

)
# 2.导入工具
web_search = TavilySearch(max_results=2)
# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)
# 4.运行Agent获得结果
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
    ]}
)

rprint(result)

{
    'messages': [
        HumanMessage(
            content='请帮我查询2026年足球世界杯是哪个国家举办的？',
            additional_kwargs={},
            response_metadata={},
            id='597c41c5-f5db-4b4e-a981-e5bb1afee18a'
        ),
        AIMessage(
            content='2026年国际足联世界杯（FIFA World Cup 2026）将由**三个国家联合举办**：\n\n- 🇺🇸 **美国**\n- 🇨🇦 
**加拿大**\n- 🇲🇽 **墨西哥**\n\n这是世界杯历史上首次由三个国家联合承办。本届赛事也将扩军至 
**48支球队**（此前为32支），比赛将在北美多个城市举行，其中美国承办大部分场次（包括决赛）。\n\n需要我帮你查询具体的
举办城市、赛程或门票信息吗？',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 107,
                    'prompt_tokens': 1830,
                    'total_tokens': 1937,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 1664,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 1664,
                    'prompt_cache_miss_tokens': 166
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                'id': '1622e50c-b53b-465b-9274-e5f7c777ad22',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a181b7-f3c6-7491-baf9-4f16cf9f9874-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1830,
                'output_tokens': 107,
                'total_tokens': 1937,
                'input_token_details': {'cache_read': 1664},
                'output_token_details': {}
            }
        )
    ]
}

举例2：

In [3]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from rich import print as rprint
# 工具：实现两数相加

@tool
def add_numbers(a: int, b: int) -> str:
    """计算并返回两个数的和。"""
    return f"和为：{a + b}"

# 创建客服助手Agent
agent = create_agent(
    model=model,
    tools=[add_numbers], # 工具列表
    # system_prompt="你是一个数学助手，解决日常的算术问题"
    system_prompt=SystemMessage(content="你是一个数学助手，解决日常的算术问题")
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "10加上20再加上30是多少？"}
    ]},
)
rprint(response)
# print(response["messages"][-1].content)

{
    'messages': [
        HumanMessage(
            content='10加上20再加上30是多少？',
            additional_kwargs={},
            response_metadata={},
            id='b01542bd-610b-477f-abe6-667c3d5c0750'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 53,
                    'prompt_tokens': 294,
                    'total_tokens': 347,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 166
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                'id': 'd6df51f4-41e9-429a-8a71-3151f69fd36f',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a181b7-ea19-7143-aaed-85c02bd0f672-0',
            tool_calls=[
                {
                    'name': 'add_numbers',
                    'args': {'a': 10, 'b': 20},
                    'id': 'call_00_qLXN3JtNgK6T6E6KjI8E4397',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 294,
                'output_tokens': 53,
                'total_tokens': 347,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='和为：30',
            name='add_numbers',
            id='7647ef6c-3dba-4541-a1ef-7cca10a8bd47',
            tool_call_id='call_00_qLXN3JtNgK6T6E6KjI8E4397'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 53,
                    'prompt_tokens': 362,
                    'total_tokens': 415,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 234
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                'id': '09ced4a6-6d2c-4760-918a-34e29a6631f4',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a181b7-edf2-7370-9087-a50f72f538ac-0',
            tool_calls=[
                {
                    'name': 'add_numbers',
                    'args': {'a': 30, 'b': 30},
                    'id': 'call_00_aIoD6PJbnoWIlxCCGVEQ7872',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 362,
                'output_tokens': 53,
                'total_tokens': 415,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='和为：60',
            name='add_numbers',
            id='ffb690ce-c64e-4584-a8e

举例3：

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage
flag = 0
@tool
def get_weather(city: str):
    """
    天气查询工具
    Args:
    city: 城市名称
    """
    global flag
    flag += 1
    if flag < 3:
        # raise Exception("暂时无法访问")
        return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"
    return f"{city}今天天气挺好"
messages = [
HumanMessage("你好，杭州今天的天气如何？")
]
agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=SystemMessage(
        "你是一个天气助手。"
        "当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，"
        "说明是临时故障，不要立即放弃；"
        "你应再次调用同一个工具，最多重试 3 次。"
        "如果 3 次后仍失败，再向用户说明服务暂时不可用。"
    )
)
response = agent.invoke({"messages": messages})
# print(response)
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，杭州今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_0ut0Exsyczx0xGdWPEak3063)
 Call ID: call_00_0ut0Exsyczx0xGdWPEak3063
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================

第一次查询失败了，我再试一次。
Tool Calls:
  get_weather (call_00_WwH9KFJqsuq7ub6EkLCH8971)
 Call ID: call_00_WwH9KFJqsuq7ub6EkLCH8971
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================

还是失败，再试最后一次。
Tool Calls:
  get_weather (call_00_BdlG58WirdOWiQXZgtvj8576)
 Call ID: call_00_BdlG58WirdOWiQXZgtvj8576
  Args:


## 3、结构化输出的四种策略

### 3.1 ProviderStrategy策略：（deepseek模型不支持）

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ProviderStrategy
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

import os
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2.模型的初始化
model = init_chat_model(
    model="deepseek:deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,

)

# 2、使用Pydantic结构化方式来定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=ProviderStrategy(ContractInfo)
)

response = agent.invoke({
    "messages":[
        {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是xiaoqing@deepseek-ai.com，电话是12345678901"}
    ]
})

rprint(response)

## 3.2 ToolStrategy策略

In [11]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import  ToolStrategy
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

import os
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2.模型的初始化
model = init_chat_model(
    model="deepseek:deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,

)

# 2、使用Pydantic结构化方式来定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContractInfo)
)

response = agent.invoke({
    "messages":[
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是xiaoqing@deepseek-ai.com，电话是12345678901"},
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是xiaoqing@deepseek-ai.com，电话是12345678901")
    ]
})

print(response["structured_response"])

name='小明' email='xiaoqing@deepseek-ai.com' phone='12345678901'


## 3、3 AutoStrategy/type策略

In [13]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy, AutoStrategy
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

import os
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2.模型的初始化
model = init_chat_model(
    model="deepseek:deepseek-chat",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,

)

# 2、使用Pydantic结构化方式来定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=AutoStrategy(ContractInfo)
)

response = agent.invoke({
    "messages":[
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是xiaoqing@deepseek-ai.com，电话是12345678901"},
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是xiaoqing@deepseek-ai.com，电话是12345678901")
    ]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息，小明的邮箱是xiaoqing@deepseek-ai.com，电话是12345678901',
            additional_kwargs={},
            response_metadata={},
            id='24f5cd29-637c-4707-86be-debaba719e22'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 75,
                    'prompt_tokens': 338,
                    'total_tokens': 413,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 128,
                        'image_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_cache_hit_tokens': 128,
                    'prompt_cache_miss_tokens': 210
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                'id': '09cbc157-73da-472e-b110-2e85e84c4b67',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a181c5-1b7a-7112-81a0-8370b115e110-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'xiaoqing@deepseek-ai.com', 'phone': '12345678901'},
                    'id': 'call_00_CinCxt3hMLlspcAaWWAQ2762',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 338,
                'output_tokens': 75,
                'total_tokens': 413,
                'input_token_details': {'cache_read': 128},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='xiaoqing@deepseek-ai.com' 
phone='12345678901'",
            name='ContractInfo',
            id='31f2f714-4bd7-41b5-812a-0e94a95ce431',
            tool_call_id='call_00_CinCxt3hMLlspcAaWWAQ2762'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='xiaoqing@deepseek-ai.com', phone='12345678901')
}